In [10]:
import pandas as pd
import numpy as np
import plotly.express as px
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import optuna
import joblib

In [11]:
df_maths = pd.read_csv(r"..\data\student-mat.csv", delimiter=";")
df_por = pd.read_csv(r"..\data\student-por.csv", delimiter=";")

df_maths['subject'] = 'Maths'
df_por['subject']= 'Portuguese'

df = pd.concat([df_maths, df_por], axis=0).reset_index(drop=True)

id_cols = ["school", "sex", "age", "address", "famsize", "Pstatus", 
           "Medu", "Fedu", "Mjob", "Fjob", "reason", "nursery", "internet"]

df['student_id'] = df[id_cols].astype(str).agg('-'.join, axis=1)

#  target
target = 'G3'

# feature sets
demographic_features = [col for col in df.columns if col not in ['G1', 'G2', 'G3', 'student_id']]

# Categorical columns 
cat_features = df[demographic_features].select_dtypes(include=['str']).columns.tolist()
for col in cat_features:
    df[col] = df[col].astype('category') 

# feature sets for  3 models
features_model1 = demographic_features
features_model2 = demographic_features + ['G1']
features_model3 = demographic_features + ['G1', 'G2']

print(df.sample(3))

    school sex  age address famsize Pstatus  Medu  Fedu      Mjob      Fjob  \
426     GP   M   15       U     GT3       T     4     4  services  services   
105     GP   F   15       U     GT3       A     3     3     other    health   
110     GP   M   15       U     LE3       A     4     4   teacher   teacher   

     ... goout Dalc  Walc  health  absences  G1  G2  G3     subject  \
426  ...     1    1     1       5         2  15  15  15  Portuguese   
105  ...     3    1     1       4        10  10  11  11       Maths   
110  ...     3    1     1       4         6  18  19  19       Maths   

                                            student_id  
426  GP-M-15-U-GT3-T-4-4-services-services-reputati...  
105  GP-F-15-U-GT3-A-3-3-other-health-reputation-ye...  
110  GP-M-15-U-LE3-A-4-4-teacher-teacher-course-yes...  

[3 rows x 35 columns]


## Model 1 (Pre-exam)

In [31]:
X = df[features_model1]
y = df['G3']
groups = df['student_id']

def objective(trial):

    params = {"tree_method": "hist","objective": "reg:squarederror","random_state": 42,
              "enable_categorical" :True, "n_estimators":1000,  #"early_stopping_rounds":50,
              
              "n_estimators": trial.suggest_int("n_estimators",200,1000),
              "max_depth": trial.suggest_int("max_depth",3,10),
              "learning_rate": trial.suggest_float("learning_rate",0.005,0.1,log=True),
              "subsample": trial.suggest_float("subsample",0.5,1.0),
              "colsample_bytree": trial.suggest_float("colsample_bytree",0.5,1.0),
              "min_child_weight": trial.suggest_int("min_child_weight",1,15),
              "gamma": trial.suggest_float("gamma",0,5),
              #"reg_alpha": trial.suggest_float("reg_alpha",1e-4,10,log=True),
              "reg_lambda": trial.suggest_float("reg_lambda",1e-4,10,log=True),
    }
    
    gkf = GroupKFold(n_splits=5)
    mae_scores = []

    for train_idx, val_idx in gkf.split(X, y, groups=groups):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**params)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))

    return np.mean(mae_scores)

In [32]:
study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=200,
    show_progress_bar=True
)

[I 2026-05-13 03:07:18,559] A new study created in memory with name: no-name-7cdbfa33-ee1e-4fb1-acb1-4f89278c2f9d


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-05-13 03:07:20,448] Trial 0 finished with value: 2.4053783893585203 and parameters: {'n_estimators': 674, 'max_depth': 3, 'learning_rate': 0.023205966112455113, 'subsample': 0.5217784317552262, 'colsample_bytree': 0.8832966017457189, 'min_child_weight': 9, 'gamma': 4.8683106792456385, 'reg_lambda': 6.746892863916153}. Best is trial 0 with value: 2.4053783893585203.
[I 2026-05-13 03:07:24,141] Trial 1 finished with value: 2.3577054500579835 and parameters: {'n_estimators': 986, 'max_depth': 6, 'learning_rate': 0.005358789970691628, 'subsample': 0.8928332436084219, 'colsample_bytree': 0.6468858080219099, 'min_child_weight': 7, 'gamma': 4.161888780565729, 'reg_lambda': 3.5473554526802182}. Best is trial 1 with value: 2.3577054500579835.
[I 2026-05-13 03:07:25,545] Trial 2 finished with value: 2.3989611625671388 and parameters: {'n_estimators': 444, 'max_depth': 6, 'learning_rate': 0.030497392854056445, 'subsample': 0.9049142311849798, 'colsample_bytree': 0.7275375575676006, 'min_c

In [33]:
trial_df = study.trials_dataframe()
fig = px.line(
    trial_df,
    x="number",
    y="value",
    title="Optuna Optimization History",template = "presentation"
)
fig.show()

In [34]:
print("Best MAE:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

Best MAE:
2.3212094783782957

Best Parameters:
{'n_estimators': 550, 'max_depth': 10, 'learning_rate': 0.006598813396461242, 'subsample': 0.5673684590681182, 'colsample_bytree': 0.5598412524097023, 'min_child_weight': 4, 'gamma': 0.6608661616089366, 'reg_lambda': 0.74289218237186}


In [36]:
def evaluate(df, feature_list, best_params, target_col='G3'):
    X = df[feature_list]
    y = df[target_col]
    groups = df['student_id']
    
    gkf = GroupKFold(n_splits=5)
    
    mae_scores = []
    mse_scores = []
    r2_scores = []
    
    print(f"\n--- Training Model with {len(feature_list)} features ---")
    
    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**best_params, tree_method= "hist",objective= "reg:squarederror",random_state= 42,
              enable_categorical =True)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))
        mse_scores.append(mean_squared_error(y_val, preds))
        r2_scores.append(r2_score(y_val, preds))
        
    print('MAE :',mae_scores)    
    print(f"Average MAE: {np.mean(mae_scores):.4f}")
    print('MSE :',mse_scores)    
    print(f"Average MSE: {np.mean(mse_scores):.4f}")
    print('r2_scores : ',r2_scores)
    print(f"Average R2: {np.mean(r2_scores):.4f}")

In [37]:
# Evaluating Model 1 (Pre-exam)
evaluate(df, features_model1,best_params=study.best_params)


--- Training Model with 31 features ---
MAE : [2.298579692840576, 2.2462849617004395, 2.3126699924468994, 2.276521921157837, 2.4719908237457275]
Average MAE: 2.3212
MSE : [9.64253044128418, 9.439459800720215, 10.167106628417969, 9.696050643920898, 11.945527076721191]
Average MSE: 10.1781
r2_scores :  [0.2961518168449402, 0.2442418336868286, 0.32159751653671265, 0.39731860160827637, 0.3027436137199402]
Average R2: 0.3124


In [38]:
# final model 1 (using the whole data)
model_1 = xgb.XGBRegressor(**study.best_params, tree_method= "hist",objective= "reg:squarederror",random_state= 42,
              enable_categorical =True)
model_1.fit(
            X, y,
            verbose=True
        )

# eval_set is must if we are using early stopping

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.5598412524097023
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

In [40]:
pred = model_1.predict(X)
mean_absolute_error(pred,y)

1.0436291694641113

### Why not pickle or joblib?
You can use joblib.dump(model_1, 'model.pkl'). It's very common. However, if you update your XGBoost version later, pickle files often break. The .save_model() method is specifically designed by the XGBoost team to be "future-proof.

In [55]:
import json

# Save the model itself
model_1.save_model("model_1.json")